<a href="https://colab.research.google.com/github/Kevan123/TrustInAI/blob/main/When_Is_An_Output_Level_Trust_Score_Meaningful_Analysis_Elsevier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

In [ ]:
%env TRACE_API_KEY=_sQT87ZJqW3oG651jQGYpBLxBqKOr8e6S8iyS6kha3g
!python run_study.py --backend https://trace-agent-backend.onrender.com --out steps.jsonl --limit 30

In [ ]:
import requests
print(requests.get("https://trace-agent-backend.onrender.com/health").json())

In [ ]:
import json
rows = [json.loads(l) for l in open("steps.jsonl")]
for r in rows[:8]:
    print(r["step_type"], r["fault_type"], r["availability"], "T=", r["T"])

In [ ]:
import requests, json
rows = [json.loads(l) for l in open("steps.jsonl")]
r = next(x for x in rows if x["step_type"] == "reason")
resp = requests.post(
    "https://trace-agent-backend.onrender.com/score_text",
    headers={"X-API-Key": "_sQT87ZJqW3oG651jQGYpBLxBqKOr8e6S8iyS6kha3g",
             "Content-Type": "application/json"},
    json={"text": r["text"], "query_anchor": r["query_anchor"],
          "step_type": "reason", "alts": r["alts"]},
    timeout=180)
print(resp.status_code)
print(resp.text[:1500])

In [ ]:
import requests
CEREBRAS_KEY = "csk-4w4vkw8xjwmdpkw4jtkncmhmj9fjnfenpc4vc6vcr63w6t62"
r = requests.get("https://api.cerebras.ai/v1/models",
                 headers={"Authorization": f"Bearer {CEREBRAS_KEY}"})
print([m["id"] for m in r.json()["data"]])

In [ ]:
import os; os.path.exists("steps.jsonl") and os.remove("steps.jsonl")
%env TRACE_API_KEY=_sQT87ZJqW3oG651jQGYpBLxBqKOr8e6S8iyS6kha3g
!python run_study.py --backend https://trace-agent-backend.onrender.com --out steps.jsonl

In [ ]:
import requests, time
for _ in range(3):
    try:
        print(requests.get("https://trace-agent-backend.onrender.com/health", timeout=30).json()); break
    except Exception as e:
        print("cold, retrying...", e); time.sleep(10)

In [ ]:
import requests
resp = requests.post("https://trace-agent-backend.onrender.com/score_text",
    headers={"X-API-Key":"_sQT87ZJqW3oG651jQGYpBLxBqKOr8e6S8iyS6kha3g","Content-Type":"application/json"},
    json={"text":"Bridgetown is the capital of Barbados, so the regional office should be located in Bridgetown.",
          "query_anchor":"What is the capital of Barbados?","step_type":"reason",
          "alts":["The capital of Barbados is Bridgetown, therefore the office belongs in Bridgetown."]},
    timeout=180)
print(resp.status_code); print(resp.text[:800])


# RERUN


In [ ]:
from google.colab import files
files.upload()   # select all five: tasks.json, faults.py, run_study.py, analyze.py, abstract_numbers.py

Saving tasks.json to tasks.json
Saving loans_deidentified.csv to loans_deidentified.csv
Saving faults.py to faults.py
Saving run_study.py to run_study.py
Saving analyze.py to analyze.py
Saving abstract_numbers.py to abstract_numbers.py


{'tasks.json': b'{\n  "tasks": [\n    {\n      "task_id": "personal_loan_eligibility",\n      "goal": "Decide whether the applicant qualifies for the personal loan under the current credit policy.",\n      "provenance": "Applicant profile derived from a de-identified row (row_id 3) of the loan dataset used in Rajaram and Hosein (2023), A Cost-Reward Optimization Approach to Bank Loan Approvals. No PII is stored; only policy-relevant numeric values are used.",\n      "kb": {\n        "policy_min_monthly_income_floor": 5000,\n        "policy_max_balance_to_loan_ratio": 0.6,\n        "policy_max_loans_90_days_delinquent": 0,\n        "applicant_income_lower_bound": 5000,\n        "applicant_total_balance": 27447.35,\n        "applicant_loan_amount": 70000,\n        "applicant_loans_90_days_delinquent": 0\n      },\n      "trajectory": [\n        {\n          "step_index": 0,\n          "step_type": "reason",\n          "text": "The credit policy sets three eligibility conditions: a minimu

In [ ]:
import json, requests
rows = [json.loads(l) for l in open("steps.jsonl")]
bad = next(r for r in rows if r.get("error") and r["step_type"] in ("reason","synthesize"))
resp = requests.post("https://trace-agent-backend.onrender.com/score_text",
    headers={"X-API-Key":"_sQT87ZJqW3oG651jQGYpBLxBqKOr8e6S8iyS6kha3g","Content-Type":"application/json"},
    json={"text":bad["text"],"query_anchor":bad["query_anchor"],
          "step_type":bad["step_type"],"alts":bad["alts"]}, timeout=300)
print("status:", resp.status_code)
print(resp.text[:1200])

status: 200
{"E":0.2301,"V":0.615,"S":0.4999,"L":1.0,"C":0.885,"T":66.52,"routing":"REVIEW","availability":{"E":"computed","V":"computed","S":"computed","L":"computed","C":"computed"},"verdict":"partial","hat_p":0.5,"step_type":"reason","n_claims":2,"contested":0}


In [ ]:
import json, collections
rows = [json.loads(l) for l in open("steps.jsonl")]
print("records:", len(rows))
print("model:", {r["model"] for r in rows})
print("failed:", sum(1 for r in rows if not isinstance(r.get("availability"), dict)))
print(collections.Counter((r["step_type"], r["condition"]) for r in rows))
for r in rows:
    if r.get("error"):
        print("ERROR example:", r["error"][:200]); break

records: 23
model: {'gpt-oss-120b'}
failed: 11
Counter({('reason', 'clean'): 4, ('act', 'faulted'): 4, ('act', 'clean'): 3, ('observe', 'clean'): 3, ('reason', 'faulted'): 3, ('synthesize', 'clean'): 2, ('observe', 'faulted'): 2, ('synthesize', 'faulted'): 2})
ERROR example: 500 Server Error: Internal Server Error for url: https://trace-agent-backend.onrender.com/score_text


In [ ]:
%env TRACE_API_KEY=_sQT87ZJqW3oG651jQGYpBLxBqKOr8e6S8iyS6kha3g
!rm -f steps.jsonl
!python run_study.py --backend https://trace-agent-backend.onrender.com --out steps.jsonl

env: TRACE_API_KEY=_sQT87ZJqW3oG651jQGYpBLxBqKOr8e6S8iyS6kha3g
wrote 41 step records to steps.jsonl (model=gpt-oss-120b)


In [ ]:
import json, collections
rows = [json.loads(l) for l in open("steps.jsonl")]
print("records:", len(rows), "| model:", {r["model"] for r in rows})
print("failed:", sum(1 for r in rows if not isinstance(r.get("availability"), dict)))
print("hat_p=0.5 share:", round(sum(1 for r in rows if r.get("hat_p")==0.5)/len(rows), 2))
print(collections.Counter((r["step_type"], r["condition"]) for r in rows))

records: 41 | model: {'gpt-oss-120b'}
failed: 0
hat_p=0.5 share: 0.0
Counter({('act', 'faulted'): 8, ('reason', 'clean'): 6, ('reason', 'faulted'): 6, ('synthesize', 'faulted'): 6, ('act', 'clean'): 4, ('observe', 'clean'): 4, ('observe', 'faulted'): 4, ('synthesize', 'clean'): 3})


In [ ]:
!python analyze.py --in steps.jsonl --outdir results
!python abstract_numbers.py --in steps.jsonl

loaded 41 step records (17 clean, 24 faulted)

== Table A: availability by step type (fraction over clean steps) ==
 step_type signal  n  computed  degenerate  not_applicable
    reason      E  6      1.00         0.0            0.00
    reason      V  6      1.00         0.0            0.00
    reason      S  6      1.00         0.0            0.00
    reason      L  6      1.00         0.0            0.00
    reason      C  6      1.00         0.0            0.00
       act      E  4      0.00         0.0            1.00
       act      V  4      0.00         0.0            1.00
       act      S  4      1.00         0.0            0.00
       act      L  4      1.00         0.0            0.00
       act      C  4      0.00         0.0            1.00
   observe      E  4      0.25         0.0            0.75
   observe      V  4      0.25         0.0            0.75
   observe      S  4      0.00         0.0            1.00
   observe      L  4      1.00         0.0            0.00

In [ ]:
import json
rows = [json.loads(l) for l in open("steps.jsonl")]
claim = [r for r in rows if isinstance(r.get("availability"),dict) and r["availability"].get("C")=="computed"]
half = sum(1 for r in claim if r.get("hat_p")==0.5)
print(f"claim steps with C computed: {len(claim)} | hat_p==0.5 degraded: {half} ({half/len(claim):.0%})" if claim else "none")
import numpy as np
for s in ["L","hat_p","C"]:
    v=[r[s] for r in rows if isinstance(r.get(s),(int,float))]
    print(f"{s}: nunique={len(set(round(x,3) for x in v))} range=[{min(v):.2f},{max(v):.2f}]")

claim steps with C computed: 23 | hat_p==0.5 degraded: 0 (0%)
L: nunique=9 range=[0.50,1.00]
hat_p: nunique=12 range=[0.00,0.99]
C: nunique=23 range=[0.27,0.99]
